In [ ]:
# Install required packages
!pip install pandas scikit-learn tensorflow keras seaborn matplotlib

In [ ]:
# ─────────────────────────────────────────────
# CELL 2 — All Imports
# ─────────────────────────────────────────────
import numpy as np
import pandas as pd
import pickle
import math
import os

import seaborn as sns
import matplotlib.pyplot as plt

from sklearn import preprocessing, metrics
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split

# Classifiers
from sklearn.ensemble import RandomForestClassifier          # FIX: was RandomForestRegressor
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis

# Metrics
from sklearn.metrics import (
    accuracy_score,
    classification_report,                                   # FIX: was regression_report (doesn't exist)
    precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, auc
)

# Deep Learning
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.models import Sequential, Model       # FIX: consolidated keras imports
from tensorflow.keras.layers import Dense, LSTM, Input      # FIX: use tensorflow.keras (not standalone keras)

print('TensorFlow version:', tf.__version__)
keras.utils.set_random_seed(42)

In [ ]:
# ─────────────────────────────────────────────
# CELL 3 — Load Dataset
# FIX: Removed extra quotes inside the path string
# FIX: Use relative paths — update filenames to where you placed the CSV files
# Dataset: https://www.unb.ca/cic/datasets/nsl.html
# ─────────────────────────────────────────────
df_train = pd.read_csv('Train_data.csv')   # <-- update path if needed
df_test  = pd.read_csv('Test_data.csv')    # <-- update path if needed

print('Train shape:', df_train.shape)
print('Test shape :', df_test.shape)

# Columns present in one but not the other
diff = set(df_train.columns) ^ set(df_test.columns)
print('Column differences between train and test:', diff)

df_train.info()

In [ ]:
# ─────────────────────────────────────────────
# CELL 4 — Exploratory Data Analysis (EDA)
# ─────────────────────────────────────────────
pd.set_option('display.max_columns', None)

numeric_cols = df_train.select_dtypes(include=np.number).columns
object_cols  = df_train.select_dtypes(include=object).columns

print(df_train.describe())

# Drop zero-variance columns (carry no information)
zero_var_cols = df_train[numeric_cols].columns[df_train[numeric_cols].std() == 0].tolist()
print(f'Zero-variance columns dropped: {zero_var_cols}')
df_train = df_train.drop(columns=zero_var_cols)
numeric_cols = numeric_cols.drop(zero_var_cols)

print('Missing values per column:')
print(df_train.isna().sum())

# Numeric feature distributions
df_train[numeric_cols].hist(figsize=(20, 15), bins=60, edgecolor='black')
plt.tight_layout()
plt.show()

In [ ]:
# ─────────────────────────────────────────────
# CELL 5 — Categorical Feature Plots
# ─────────────────────────────────────────────
# FIX: Added guard so plotting only runs if there are categorical columns
cat_cols_to_plot = [c for c in object_cols if c != 'class']

if cat_cols_to_plot:
    graph_cols = 2
    nrows = math.ceil(len(cat_cols_to_plot) / graph_cols)
    fig, axes = plt.subplots(nrows=nrows, ncols=graph_cols,
                             figsize=(8 * graph_cols, 5 * nrows))
    axes = np.array(axes).reshape(nrows, graph_cols)  # ensure 2D

    for ix, col in enumerate(cat_cols_to_plot):
        i, j = ix // graph_cols, ix % graph_cols
        value_counts = df_train[col].value_counts()
        sns.barplot(x=value_counts.index, y=value_counts.values,
                    palette='pastel', ax=axes[i][j])
        axes[i][j].set_title(f'Value Count for {col}')
        axes[i][j].set_ylabel('Count')
        axes[i][j].tick_params(axis='x', rotation=45)

    plt.tight_layout()
    plt.show()

# Class distribution
print('Class distribution:')
print(df_train['class'].value_counts())

In [ ]:
# ─────────────────────────────────────────────
# CELL 6 — Preprocessing
# ─────────────────────────────────────────────
# FIX: Removed duplicate 'Date and Time' drop (column doesn't exist in NSL-KDD)
# FIX: One-hot encode categorical features; label-encode target

TARGET = 'class'

# Separate features and target
X = df_train.drop(columns=[TARGET])
y = df_train[TARGET]

# Encode target: 'normal' -> 0, 'anomaly' -> 1
le = LabelEncoder()
y = le.fit_transform(y)
print('Classes:', le.classes_)

# One-hot encode categorical feature columns
X = pd.get_dummies(X)

# Align test set columns to match training (fill missing cols with 0)
X_test_raw = df_test.drop(columns=[TARGET], errors='ignore')
X_test_raw = pd.get_dummies(X_test_raw)
X_test_raw = X_test_raw.reindex(columns=X.columns, fill_value=0)

# Train / validation split
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled   = scaler.transform(X_val)           # FIX: transform only, no fit
X_test_scaled  = scaler.transform(X_test_raw)      # FIX: transform only, no fit

print(f'Train: {X_train_scaled.shape}  Val: {X_val_scaled.shape}  Test: {X_test_scaled.shape}')

In [ ]:
# ─────────────────────────────────────────────
# CELL 7 — Train & Compare Classical Classifiers
# FIX: All models are classifiers, not regressors
# FIX: accuracy_score now correctly used with classifier output
# ─────────────────────────────────────────────
classifiers = {
    'Random Forest' : RandomForestClassifier(n_estimators=100, random_state=42),
    'SVM'           : SVC(kernel='rbf', probability=True, random_state=42),
    'KNN'           : KNeighborsClassifier(n_neighbors=5),
    'LDA'           : LinearDiscriminantAnalysis(),
    'QDA'           : QuadraticDiscriminantAnalysis(),
}

results = {}
trained_models = {}

for name, clf in classifiers.items():
    clf.fit(X_train_scaled, y_train)
    y_pred = clf.predict(X_val_scaled)
    acc  = accuracy_score(y_val, y_pred)
    f1   = f1_score(y_val, y_pred, average='weighted')
    results[name] = {'Accuracy': round(acc, 4), 'F1 Score': round(f1, 4)}
    trained_models[name] = clf
    print(f'{name:20s} → Accuracy: {acc:.4f}  F1: {f1:.4f}')

print('\nBest model:', max(results, key=lambda k: results[k]['Accuracy']))

In [ ]:
# ─────────────────────────────────────────────
# CELL 8 — Classification Report & ROC Curve
# FIX: classification_report replaces non-existent regression_report
# ─────────────────────────────────────────────
best_name = max(results, key=lambda k: results[k]['Accuracy'])
best_model = trained_models[best_name]
y_pred_best = best_model.predict(X_val_scaled)

print(f'=== {best_name} — Classification Report ===')
print(classification_report(y_val, y_pred_best, target_names=le.classes_))

# ROC Curve (only if model supports predict_proba)
if hasattr(best_model, 'predict_proba'):
    y_prob = best_model.predict_proba(X_val_scaled)[:, 1]
    fpr, tpr, _ = roc_curve(y_val, y_prob)
    roc_auc = auc(fpr, tpr)

    plt.figure(figsize=(7, 5))
    plt.plot(fpr, tpr, color='darkorange', lw=2,
             label=f'ROC Curve (AUC = {roc_auc:.4f})')
    plt.plot([0, 1], [0, 1], color='navy', lw=1, linestyle='--')
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title(f'ROC Curve — {best_name}')
    plt.legend(loc='lower right')
    plt.tight_layout()
    plt.show()

In [ ]:
# ─────────────────────────────────────────────
# CELL 9 — Deep Learning: Sequential Dense Network
# FIX: Consolidated keras imports from tensorflow.keras
# FIX: venv activation inside notebook has no effect — removed misleading shell commands
# ─────────────────────────────────────────────
n_features = X_train_scaled.shape[1]

dense_model = Sequential([
    Dense(128, activation='relu', input_shape=(n_features,)),
    Dense(64,  activation='relu'),
    Dense(32,  activation='relu'),
    Dense(1,   activation='sigmoid')   # binary classification
])

dense_model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)
dense_model.summary()

history = dense_model.fit(
    X_train_scaled, y_train,
    epochs=20,
    batch_size=256,
    validation_data=(X_val_scaled, y_val),
    verbose=1
)

# Plot training history
plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'],     label='Train Accuracy')
plt.plot(history.history['val_accuracy'], label='Val Accuracy')
plt.title('Dense Model — Accuracy')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history['loss'],     label='Train Loss')
plt.plot(history.history['val_loss'], label='Val Loss')
plt.title('Dense Model — Loss')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ─────────────────────────────────────────────
# CELL 10 — Deep Learning: LSTM Network
# FIX: LSTM requires 3D input — reshape added
# ─────────────────────────────────────────────
# Reshape for LSTM: (samples, timesteps=1, features)
X_train_lstm = X_train_scaled.reshape(X_train_scaled.shape[0], 1, X_train_scaled.shape[1])
X_val_lstm   = X_val_scaled.reshape(X_val_scaled.shape[0], 1, X_val_scaled.shape[1])

lstm_model = Sequential([
    LSTM(64, input_shape=(1, n_features), return_sequences=False),
    Dense(32, activation='relu'),
    Dense(1,  activation='sigmoid')
])

lstm_model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)
lstm_model.summary()

lstm_history = lstm_model.fit(
    X_train_lstm, y_train,
    epochs=15,
    batch_size=256,
    validation_data=(X_val_lstm, y_val),
    verbose=1
)

In [ ]:
# ─────────────────────────────────────────────
# CELL 11 — Save Best Classical Model
# ─────────────────────────────────────────────
with open('best_model.pkl', 'wb') as f:
    pickle.dump(best_model, f)
print(f'Saved best classical model ({best_name}) to best_model.pkl')

# Save deep learning model
dense_model.save('dense_intrusion_model.keras')
print('Saved Dense model to dense_intrusion_model.keras')